# Clickstream Analytics Pipeline 



In [2]:
pip install pyspark delta-spark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 2.8 MB/s  0:02:440:00:0100:05
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 2.3 MB/s  0:04:170:00:0100:06
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008704 sha256=cc66a2117d41fdb34fa59dcce1f13f55c7ed9a7cdaafa4711788eab78c305971
  Stored in directory: /Users/vahramdressler/Library/Caches/pip/wheels/16/33/a9/f8bff354a182417214933df74dace2a34b02c3e5643e8fac74
Successfully built pyspark
  Attempting uninstall: importlib_metadata━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [pyspark]
    Found existing installation: importlib_metadata 9.0.0━━━━━ 1/4 [pyspark]
    Uninstalling importlib_metadata-9.0.0:━━━━━━━━━━━━━━━━━━━━ 1/4 [pysp

In [7]:
import os
import pyspark
from pyspark.sql import SparkSession

# Delta Lake requires specific configuration
spark = SparkSession.builder \
    .appName("ClickstreamPipeline") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0") \
    .config("spark.sql.shuffle.partitions", "4") \
    .master("local[*]") \
    .getOrCreate()

print(f"PySpark version: {spark.version}")
print("✅ Spark session ready with Delta Lake support")

PySpark version: 4.1.1
✅ Spark session ready with Delta Lake support


26/05/22 21:31:28 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [8]:
# Create directories for data and checkpoints
dirs = [
    'data/raw', 'data/bronze', 'data/silver', 'data/gold',
    'checkpoints/bronze', 'checkpoints/silver'
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
print('Directories created')

Directories created


In [11]:
# Generate fake clickstream data (simulates Kafka messages)
import json as json_mod
import uuid
import random
from datetime import datetime, timedelta

USERS = [f'user_{i:04d}' for i in range(1, 201)]
PAGES = [
    '/home', '/products', '/products/electronics',
    '/products/electronics/laptop-pro', '/cart', '/checkout',
    '/search', '/products/clothing', '/products/books'
]
EVENT_TYPES = ['page_view', 'add_to_cart', 'search', 'purchase', 'remove_from_cart']

PRODUCTS = {
    '/products/electronics/laptop-pro': {'id': 'prod_001', 'price': 1299.99, 'category': 'electronics'},
    '/products/clothing': {'id': 'prod_002', 'price': 89.99, 'category': 'clothing'},
    '/products/books': {'id': 'prod_003', 'price': 39.99, 'category': 'books'},
}

def generate_event(user_id, ts):
    page = random.choice(PAGES)
    etype = random.choice(EVENT_TYPES)
    product = PRODUCTS.get(page, {})
    
    event = {
        'event_id': str(uuid.uuid4()),
        'user_id': user_id,
        'session_id': None,
        'event_type': etype,
        'page_url': page,
        'referrer_url': random.choice(PAGES) if random.random() > 0.5 else None,
        'product_id': product.get('id'),
        'timestamp': ts.isoformat() + 'Z',
        'user_agent': random.choice([
            'Mozilla/5.0 Chrome/120.0',
            'Mozilla/5.0 Safari/605.1',
            'Mozilla/5.0 Mobile Safari/16.0'
        ]),
        'ip_address': f'10.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}',
        'metadata': {
            'price': product.get('price'),
            'category': product.get('category'),
            'search_query': f'best {random.choice(["laptop","jacket","book"])}' if etype == 'search' else None
        }
    }
    return event

def generate_session(user_id, start_time):
    events = []
    current_ts = start_time
    num_events = random.randint(2, 10)
    for _ in range(num_events):
        events.append(generate_event(user_id, current_ts))
        current_ts += timedelta(seconds=random.randint(10, 300))
    return events

def write_batch(batch_id, start_time, num_sessions=20):
    all_events = []
    for _ in range(num_sessions):
        uid = random.choice(USERS)
        offset = timedelta(seconds=random.randint(0, 300))
        all_events.extend(generate_session(uid, start_time + offset))
    all_events.sort(key=lambda e: e['timestamp'])
    filepath = f'data/raw/batch_{batch_id:04d}.jsonl'
    with open(filepath, 'w') as f:
        for event in all_events:
            f.write(json_mod.dumps(event) + '\n')
    return len(all_events)

base_time = datetime(2024, 3, 15, 10, 0, 0)
total_events = 0
for i in range(1, 11):
    t = base_time + timedelta(minutes=i * 5)
    n = write_batch(i, t, num_sessions=15)
    total_events += n
    print(f'Batch {i:02d}: {n} events')
print(f'Total events generated: {total_events}')

Batch 01: 86 events
Batch 02: 85 events
Batch 03: 104 events
Batch 04: 72 events
Batch 05: 96 events
Batch 06: 87 events
Batch 07: 81 events
Batch 08: 108 events
Batch 09: 81 events
Batch 10: 70 events
Total events generated: 870


In [12]:
# Bronze Layer: Stream JSON files into Delta table
from pyspark.sql.types import *
from pyspark.sql.functions import col, current_timestamp, to_timestamp, input_file_name, lit

clickstream_schema = StructType([
    StructField('event_id', StringType(), False),
    StructField('user_id', StringType(), False),
    StructField('session_id', StringType(), True),
    StructField('event_type', StringType(), False),
    StructField('page_url', StringType(), True),
    StructField('referrer_url', StringType(), True),
    StructField('product_id', StringType(), True),
    StructField('timestamp', StringType(), False),
    StructField('user_agent', StringType(), True),
    StructField('ip_address', StringType(), True),
    StructField('metadata', StructType([
        StructField('price', DoubleType(), True),
        StructField('category', StringType(), True),
        StructField('search_query', StringType(), True)
    ]))
])

raw_stream = spark.readStream \
    .format('json') \
    .schema(clickstream_schema) \
    .option('maxFilesPerTrigger', 2) \
    .load('data/raw')

bronze_with_meta = raw_stream \
    .withColumn('event_timestamp', to_timestamp(col('timestamp'))) \
    .withColumn('ingestion_time', current_timestamp()) \
    .withColumn('source_file', input_file_name()) \
    .withColumn('pipeline_version', lit('v1.0'))

bronze_query = bronze_with_meta.writeStream \
    .format('delta') \
    .outputMode('append') \
    .option('checkpointLocation', 'checkpoints/bronze') \
    .trigger(processingTime='10 seconds') \
    .start('data/bronze')

print('Bronze streaming started...')
import time
time.sleep(30)
bronze_query.stop()
print(f'Bronze rows: {spark.read.format("delta").load("data/bronze").count()}')

Py4JJavaError: An error occurred while calling o115.start.
: org.apache.spark.SparkClassNotFoundException: [DATA_SOURCE_NOT_FOUND] Failed to find the data source: delta. Make sure the provider name is correct and the package is properly registered and compatible with your Spark version. SQLSTATE: 42K02
	at org.apache.spark.sql.errors.QueryExecutionErrors$.dataSourceNotFoundError(QueryExecutionErrors.scala:764)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:686)
	at org.apache.spark.sql.classic.DataStreamWriter.startInternal(DataStreamWriter.scala:249)
	at org.apache.spark.sql.classic.DataStreamWriter.start(DataStreamWriter.scala:132)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:75)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:52)
	at java.base/java.

In [ ]:
# Silver Layer: Sessionization using window functions
from pyspark.sql.functions import lag, unix_timestamp, sum as spark_sum, when, concat, lit, trim, lower, col
from pyspark.sql.window import Window

bronze_df = spark.read.format('delta').load('data/bronze')

# Clean and deduplicate
cleaned_df = bronze_df \
    .withColumn('event_type', lower(trim(col('event_type')))) \
    .filter(col('user_id').isNotNull() & col('event_timestamp').isNotNull()) \
    .dropDuplicates(['event_id'])

# Sessionization algorithm
SESSION_GAP_SECONDS = 30 * 60
user_window = Window.partitionBy('user_id').orderBy('event_timestamp')

sessionized = cleaned_df \
    .withColumn('prev_timestamp', lag('event_timestamp', 1).over(user_window)) \
    .withColumn('gap_seconds',
        when(col('prev_timestamp').isNull(), 0)
        .otherwise(unix_timestamp('event_timestamp') - unix_timestamp('prev_timestamp'))
    ) \
    .withColumn('is_new_session', when(col('gap_seconds') > SESSION_GAP_SECONDS, 1).otherwise(0)) \
    .withColumn('session_number', spark_sum('is_new_session').over(user_window)) \
    .withColumn('session_id_derived',
        concat(col('user_id'), lit('_session_'), col('session_number').cast('string'))
    ) \
    .drop('prev_timestamp', 'gap_seconds', 'is_new_session', 'session_number')

# Write to Silver Delta table
sessionized.write.format('delta').mode('overwrite') \
    .save('data/silver')

print(f'Silver rows: {spark.read.format("delta").load("data/silver").count()}')
print(f'Sessions created: {sessionized.select("session_id_derived").distinct().count()}')

In [ ]:
# Gold Layer: Star schema with session facts and user dimensions
silver_df = spark.read.format('delta').load('data/silver')
silver_df.createOrReplaceTempView('silver')

# Session facts table
session_facts = spark.sql('''
    SELECT
        session_id_derived AS session_id,
        user_id,
        MIN(event_timestamp) AS session_start,
        MAX(event_timestamp) AS session_end,
        COUNT(*) AS total_events,
        COUNT(DISTINCT page_url) AS unique_pages_visited,
        SUM(CASE WHEN event_type = 'add_to_cart' THEN 1 ELSE 0 END) AS cart_additions,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases,
        SUM(CASE WHEN event_type = 'search' THEN 1 ELSE 0 END) AS searches,
        SUM(CASE WHEN event_type = 'purchase' AND metadata.price IS NOT NULL
                 THEN metadata.price ELSE 0 END) AS total_revenue,
        MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS converted,
        CAST((unix_timestamp(MAX(event_timestamp)) - unix_timestamp(MIN(event_timestamp))) / 60.0
             AS DOUBLE) AS session_duration_minutes
    FROM silver
    GROUP BY session_id_derived, user_id
''')

session_facts.write.format('delta').mode('overwrite').save('data/gold/session_facts')

# User dimension table
user_dim = spark.sql('''
    SELECT
        user_id,
        MIN(session_start) AS first_seen,
        MAX(session_start) AS last_seen,
        COUNT(*) AS lifetime_sessions,
        SUM(total_revenue) AS lifetime_revenue,
        SUM(purchases) AS lifetime_purchases,
        CASE
            WHEN SUM(total_revenue) > 1000 THEN 'high_value'
            WHEN SUM(total_revenue) > 200  THEN 'medium_value'
            WHEN SUM(purchases) > 0          THEN 'purchaser'
            ELSE 'browser'
        END AS user_segment
    FROM (SELECT user_id, session_start, total_revenue, purchases FROM session_facts)
    GROUP BY user_id
''')

user_dim.write.format('delta').mode('overwrite').save('data/gold/dim_users')

print(f'Gold session facts: {spark.read.format("delta").load("data/gold/session_facts").count()} rows')
print(f'Gold user dimensions: {spark.read.format("delta").load("data/gold/dim_users").count()} rows')

In [ ]:
# Analytical Queries

# Query 1: Conversion funnel
print('=' * 60)
print('QUERY 1: Conversion Funnel')
print('=' * 60)
spark.sql('''
    SELECT
        COUNT(DISTINCT session_id) AS total_sessions,
        COUNT(DISTINCT CASE WHEN cart_additions > 0 THEN session_id END) AS sessions_with_cart,
        COUNT(DISTINCT CASE WHEN purchases > 0 THEN session_id END) AS sessions_with_purchase,
        ROUND(COUNT(DISTINCT CASE WHEN purchases > 0 THEN session_id END) * 100.0 /
              COUNT(DISTINCT session_id), 1) AS conversion_rate_pct,
        ROUND(SUM(total_revenue), 2) AS total_revenue
    FROM delta.`data/gold/session_facts`
''').show()

# Query 2: User segments
print('=' * 60)
print('QUERY 2: User Segments')
print('=' * 60)
spark.sql('''
    SELECT user_segment, COUNT(*) AS users, ROUND(SUM(lifetime_revenue), 2) AS total_rev
    FROM delta.`data/gold/dim_users`
    GROUP BY user_segment
    ORDER BY total_rev DESC
''').show()

# Query 3: Top landing pages
print('=' * 60)
print('QUERY 3: Session Metrics')
print('=' * 60)
spark.sql('''
    SELECT
        CASE
            WHEN session_duration_minutes < 1  THEN 'Under 1 min'
            WHEN session_duration_minutes < 5  THEN '1-5 min'
            WHEN session_duration_minutes < 15 THEN '5-15 min'
            ELSE 'Over 15 min'
        END AS duration_bucket,
        COUNT(*) AS sessions,
        ROUND(AVG(total_revenue), 2) AS avg_revenue
    FROM delta.`data/gold/session_facts`
    GROUP BY duration_bucket
    ORDER BY avg_revenue DESC
''').show()

In [ ]:
# Delta Lake Advanced Features

print('Time Travel: View table history')
print('=' * 60)
spark.sql('DESCRIBE HISTORY delta.`data/gold/session_facts`').show(truncate=False)

print('Schema Evolution: Add test column')
print('=' * 60)
spark.sql('ALTER TABLE delta.`data/gold/session_facts` ADD COLUMN test_feature STRING')
print('Column added successfully')

print('OPTIMIZE with ZORDER')
print('=' * 60)
spark.sql('OPTIMIZE delta.`data/gold/session_facts` ZORDER BY (user_id)')
print('Table optimized')

In [ ]:
# Data Quality Checks
print('Data Quality Checks')
print('=' * 60)

checks = [
    ('No null event_ids', 'SELECT COUNT(*) as fails FROM delta.`data/bronze` WHERE event_id IS NULL'),
    ('No null user_ids', 'SELECT COUNT(*) as fails FROM delta.`data/bronze` WHERE user_id IS NULL'),
    ('Non-negative revenue', 'SELECT COUNT(*) as fails FROM delta.`data/gold/session_facts` WHERE total_revenue < 0'),
    ('Valid session durations', 'SELECT COUNT(*) as fails FROM delta.`data/gold/session_facts` WHERE session_duration_minutes < 0'),
]

for check_name, query in checks:
    result = spark.sql(query).collect()[0][0]
    status = 'PASSED' if result == 0 else f'FAILED ({result} rows)'
    print(f'  {status}: {check_name}')